# Market A Simulation — Phase 1

**Unit tests** (Cases 1-4): isolated scenarios to verify fill logic and hedge routing.  
**Controller demo**: end-to-end simulation using the `Controller` class — validates all new components
(PnLTracker, per-trade MtM, fill-rate analysis, inventory management, emergency quotes).

Demo uses `dt = 5 s`, 1 trading day → 17 280 steps (~5 min runtime).

In [1]:
import sys, copy
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append('../src')

# ── reload all modules so edits are picked up without restarting the kernel ──
import utils.stock_simulation as stock_mod
import utils.market_simulator as market_mod
import utils.order_book.order_book_impl as book_mod
import utils.order_book.graphic_utils as graph_utils
import utils.market_maker.quoter as quoter_mod
import utils.report.pnl_tracker as pnl_mod
import utils.report.controller as ctrl_mod
import utils.client_flow.flow_generator as flow_mod

for mod in [stock_mod, market_mod, book_mod, graph_utils,
            quoter_mod, pnl_mod, ctrl_mod, flow_mod]:
    importlib.reload(mod)

from utils.stock_simulation import Stock
from utils.market_simulator.market import Market
from utils.order_book.order_book_impl import Order_book, Order, generate_order_id
from utils.order_book.graphic_utils import plot_order_book
from utils.market_maker.quoter import Quoter, QuoterConfig
from utils.report.pnl_tracker import PnLTracker
from utils.report.controller import Controller
from utils.client_flow.flow_generator import ClientFlowGenerator

SEED = 42
np.random.seed(SEED)
print('All modules loaded.')

All modules loaded.


## Shared setup — stock & markets for unit-test cases

Cases 1-4 only exercise 1-2 steps, so we keep the high-frequency `dt = 0.01 s` path to
preserve exact price levels.  `generate_depth()` is called so Case 4's hedge uses real
depth values instead of the fallback.

In [2]:
stock = Stock(drift=0.0, vol=0.20)
stock.simulate_garch(n_days=1, dt_seconds=0.01,
                     alpha=0.05, beta=0.94, lam=756, sigma_J=0.005)

market_B = Market(stock=stock)
market_B.generate_noised_mid_price()
market_B.build_spread(option='Skew', window_size=600, alpha=0.5,
                      gamma=0.3, ema_span=500, threshold=3)
market_B.generate_depth(mean_eur=500_000)

market_C = copy.deepcopy(market_B)
market_C.build_spread(option='Adaptive', window_size=600)
market_C.generate_depth(mean_eur=200_000)

dt      = stock.time_step
CAPITAL = 1_000_000.0
cfg     = QuoterConfig(requote_threshold_spread_fraction=0.25)


def new_sim():
    book = Order_book()
    mm   = Quoter(market_B, market_C, config=cfg, capital_K=CAPITAL)
    book.register_quoter_listener(mm.on_fill)
    return book, mm


def post_initial_ladder(book, mm, step=0):
    book.tick(step)
    quotes, cancels = mm.compute_quotes(step, step * dt, book.mm_resting_orders)
    book.cancel_orders(cancels)
    book.post_mm_quotes(quotes)
    print(f'  Ladder posted: {len(quotes)} orders')


def show_history(mm):
    df = mm.trade_history
    if df.empty:
        print('  No trades yet.')
        return
    cols = ['step', 'direction', 'level', 'price', 'size',
            'is_full_fill', 'cash_flow', 'inventory_after', 'is_hedge', 'venue']
    print(df[cols].to_string(index=False))
    print(f'\nInventory: {mm.inventory:.0f} EUR')


def show_pnl(mm):
    df = mm.trade_history
    if df.empty:
        return
    mid = float(df['fair_mid'].iloc[-1])
    rep = PnLTracker.report(df, mid)
    print(f'  Realized P&L  : {rep["realized_pnl"]:+.4f} USD')
    print(f'  MtM P&L       : {rep["total_mtm_pnl"]:+.4f} USD')
    print(f'  Inception     : {rep["inception_spread_pnl"]:+.4f} USD')
    print(f'  Revaluation   : {rep["inventory_revaluation_pnl"]:+.4f} USD')
    print(f'  Total fees    : {rep["total_fees"]:.4f} USD')

Coefficient annualization : 46661.333028536596


## Case 1 — Full fill only
Client buys the entire level-1 ask → slot gone → re-quoted at the next step.

In [3]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

resting = book.mm_resting_orders
ask_l1_price, ask_l1_id = min(
    (v['price'], oid) for oid, v in resting.items() if v['direction'] == 'sell')
ask_l1_size = resting[ask_l1_id]['original_size']
print(f'  Level-1 ask: price={ask_l1_price:.5f}, size={ask_l1_size}')

book.route_client_order(
    Order(generate_order_id(), 'buy', ask_l1_price, ask_l1_size, 'limit_order', 'client'))
print(f'  After fill — _pending_fills: {len(mm._pending_fills)}, '
      f'_pending_topups: {len(mm._pending_topups)}')

book.tick(1)
quotes, cancels = mm.compute_quotes(1, 1 * dt, book.mm_resting_orders)
book.cancel_orders(cancels)
book.post_mm_quotes(quotes)
print(f'  Requote step: {len(quotes)} new order(s), {len(cancels)} cancel(s)\n')
show_history(mm)
print()
show_pnl(mm)

  Ladder posted: 20 orders
  Level-1 ask: price=100.00880, size=74082
  After fill — _pending_fills: 1, _pending_topups: 0
  Requote step: 20 new order(s), 19 cancel(s)

 step direction  level    price  size  is_full_fill    cash_flow  inventory_after  is_hedge venue
    0      sell      1 100.0088 74082          True 7.408111e+06         -74082.0     False     A

Inventory: -74082 EUR

  Realized P&L  : +7408111.0364 USD
  MtM P&L       : -89.0186 USD
  Inception     : +651.8666 USD
  Revaluation   : -0.0000 USD
  Total fees    : 740.8852 USD


## Case 2 — Partial fill only
Client sells half of the level-1 bid → order still resting at reduced size → top-up posted.

In [4]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

resting = book.mm_resting_orders
bid_l1_price, bid_l1_id = max(
    (v['price'], oid) for oid, v in resting.items() if v['direction'] == 'buy')
bid_l1_size  = resting[bid_l1_id]['original_size']
partial_size = bid_l1_size // 2
print(f'  Level-1 bid: price={bid_l1_price:.5f}, size={bid_l1_size}')
print(f'  Sending partial sell: {partial_size}')

book.route_client_order(
    Order(generate_order_id(), 'sell', bid_l1_price, partial_size, 'limit_order', 'client'))
remaining = book.mm_resting_orders.get(bid_l1_id, {}).get('remaining_size', 'gone')
print(f'  Remaining on that order: {remaining}  (original: {bid_l1_size})')
print(f'  After fill — _pending_fills: {len(mm._pending_fills)}, '
      f'_pending_topups: {len(mm._pending_topups)}')

book.tick(1)
quotes, cancels = mm.compute_quotes(1, 1 * dt, book.mm_resting_orders)
book.cancel_orders(cancels)
book.post_mm_quotes(quotes)
print(f'  Top-up step: {len(quotes)} new order(s), {len(cancels)} cancel(s)\n')
show_history(mm)
print()
show_pnl(mm)

  Ladder posted: 20 orders
  Level-1 bid: price=99.99120, size=74082
  Sending partial sell: 37041
  Remaining on that order: 37041  (original: 74082)
  After fill — _pending_fills: 0, _pending_topups: 1
  Top-up step: 20 new order(s), 20 cancel(s)

 step direction  level   price  size  is_full_fill     cash_flow  inventory_after  is_hedge venue
    0       buy      1 99.9912 37041         False -3.704144e+06          37041.0     False     A

Inventory: 37041 EUR

  Realized P&L  : -3704144.4166 USD
  MtM P&L       : -44.3891 USD
  Inception     : +325.9883 USD
  Revaluation   : -0.0000 USD
  Total fees    : 370.3774 USD


## Case 3 — Full fill + partial fill in the same step
Two client orders arrive: one fully consumes ask L1, one partially fills bid L1.
Both handled in one Priority-4 pass: re-quote + top-up emitted together.

In [5]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

resting = book.mm_resting_orders
ask_l1_price, ask_l1_id = min(
    (v['price'], oid) for oid, v in resting.items() if v['direction'] == 'sell')
ask_l1_size  = resting[ask_l1_id]['original_size']
bid_l1_price, bid_l1_id = max(
    (v['price'], oid) for oid, v in resting.items() if v['direction'] == 'buy')
bid_l1_size  = resting[bid_l1_id]['original_size']
partial_size = bid_l1_size // 2

print(f'  Full fill  → client buy  {ask_l1_size} @ {ask_l1_price:.5f} (ask L1)')
print(f'  Partial    → client sell {partial_size} @ {bid_l1_price:.5f} (bid L1, half size)')

book.route_client_order(
    Order(generate_order_id(), 'buy',  ask_l1_price, ask_l1_size,  'limit_order', 'client'))
book.route_client_order(
    Order(generate_order_id(), 'sell', bid_l1_price, partial_size, 'limit_order', 'client'))
print(f'  After fills — _pending_fills: {len(mm._pending_fills)}, '
      f'_pending_topups: {len(mm._pending_topups)}')

book.tick(1)
quotes, cancels = mm.compute_quotes(1, 1 * dt, book.mm_resting_orders)
book.cancel_orders(cancels)
book.post_mm_quotes(quotes)
print(f'  Requote + top-up: {len(quotes)} new order(s), {len(cancels)} cancel(s)\n')
show_history(mm)
print()
show_pnl(mm)

  Ladder posted: 20 orders
  Full fill  → client buy  74082 @ 100.00880 (ask L1)
  Partial    → client sell 37041 @ 99.99120 (bid L1, half size)
  After fills — _pending_fills: 1, _pending_topups: 1
  Requote + top-up: 20 new order(s), 19 cancel(s)

 step direction  level    price  size  is_full_fill     cash_flow  inventory_after  is_hedge venue
    0      sell      1 100.0088 74082          True  7.408111e+06         -74082.0     False     A
    0       buy      1  99.9912 37041         False -3.704144e+06         -37041.0     False     A

Inventory: -37041 EUR

  Realized P&L  : +3703966.6198 USD
  MtM P&L       : -133.4077 USD
  Inception     : +977.8549 USD
  Revaluation   : -0.0000 USD
  Total fees    : 1111.2626 USD


## Case 4 — Forced hedge (with real depth from `market_B.depth`)

Inventory is pre-loaded to 91% of capital (above the 90% `delta_limit`).
`execute_hedge()` reads actual venue depth from the depth array, splits optimally across
B/C, and fills at the **venue bid/ask** (not fair_mid) — taker spread cost included.

In [6]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

mm.inventory = CAPITAL * 0.91
print(f'  Inventory pre-loaded : {mm.inventory:.0f} EUR  '
      f'({mm.inventory / CAPITAL:.0%} of capital)')
print(f'  needs_hedge          : {mm.needs_hedge()}')

resting   = book.mm_resting_orders
bids      = [v['price'] for v in resting.values() if v['direction'] == 'buy']
asks      = [v['price'] for v in resting.values() if v['direction'] == 'sell']
mid       = (max(bids) + min(asks)) / 2.0

depth_B_step0 = float(market_B.depth[0])
depth_C_step0 = float(market_C.depth[0])
print(f'  Depth available B    : {depth_B_step0:,.0f} EUR (step 0)')
print(f'  Depth available C    : {depth_C_step0:,.0f} EUR (step 0)')

hedged = mm.execute_hedge(step=0, t=0.0, fair_mid=mid)
print(f'  Hedge executed       : {hedged}')
print(f'  Inventory after      : {mm.inventory:.2f} EUR  '
      f'({mm.inventory / CAPITAL:.2%} of capital)\n')
show_history(mm)
print()
show_pnl(mm)

  Ladder posted: 20 orders
  Inventory pre-loaded : 910000 EUR  (91% of capital)
  needs_hedge          : True
  Depth available B    : 465,187 EUR (step 0)
  Depth available C    : 200,223 EUR (step 0)
  Hedge executed       : True
  Inventory after      : 313396.85 EUR  (31.34% of capital)

 step direction  level     price      size  is_full_fill    cash_flow  inventory_after  is_hedge venue
    0      sell      0 99.355422 396380.37          True 3.937466e+07        513619.63      True     B
    0      sell      0 99.500001 200222.78          True 1.991619e+07        313396.85      True     C

Inventory: 313397 EUR

  Realized P&L  : +59290852.4582 USD
  MtM P&L       : +90630537.4582 USD
  Inception     : +0.0000 USD
  Revaluation   : +90644390.6160 USD
  Total fees    : 13853.1578 USD


---
## Controller Demo Simulation

End-to-end backtest using the new `Controller` class.  
Parameters: `dt = 300 s`, 1 trading day → **288 steps**.

The `Controller`:
- wraps the step loop (tick → compute_quotes → cancel → post → client_flow → execute_hedge)
- logs every step to a lightweight DataFrame (`ctrl.step_log`)
- exposes `ctrl.report()` which prints the P&L breakdown and renders all 6 charts

In [9]:
# ── Build price path ────────────────────────────────────────────────────────
np.random.seed(SEED)

demo_stock = Stock(drift=0.0, vol=0.20)
demo_stock.simulate_garch(n_days=1, dt_seconds=3600.0,
                          alpha=0.05, beta=0.94, lam=756, sigma_J=0.005)
print(f'Stock path: {demo_stock.n_steps} steps, dt = {demo_stock.time_step:.1f} s')

# ── Build reference markets — depth derived from spread width ────────────────
mkt_B = Market(stock=demo_stock)
mkt_B.generate_noised_mid_price()
mkt_B.build_spread(option='Skew', window_size=120, alpha=0.5,
                   gamma=0.3, ema_span=100, threshold=3)
mkt_B.generate_depth(mean_eur=500_000)   # depth ∝ 1 / spread[t]

mkt_C = copy.deepcopy(mkt_B)
mkt_C.build_spread(option='Adaptive', window_size=120)
mkt_C.generate_depth(mean_eur=200_000)

print(f'Depth B: mean = {mkt_B.depth.mean():,.0f} EUR, '
      f'min = {mkt_B.depth.min():,.0f}, max = {mkt_B.depth.max():,.0f}')
print(f'Depth C: mean = {mkt_C.depth.mean():,.0f} EUR, '
      f'min = {mkt_C.depth.min():,.0f}, max = {mkt_C.depth.max():,.0f}')

# ── Build Quoter and Order_book ──────────────────────────────────────────────
demo_cfg = QuoterConfig(
    requote_threshold_spread_fraction=0.25,
    delta_limit=0.90,
    hedge_partial_limit=0.80,
    emergency_penalty_multiplier=5.0,
)

demo_book = Order_book()
demo_mm   = Quoter(mkt_B, mkt_C, config=demo_cfg, capital_K=CAPITAL)
demo_book.register_quoter_listener(demo_mm.on_fill)

# ── Client flow generator wrapped for Controller signature ───────────────────
gen = ClientFlowGenerator(seed=SEED)
client_flow_fn = lambda step, t, mid, bid, ask, dt: \
    gen.generate_step(mid_price=mid, best_bid=bid, best_ask=ask, dt=dt)

# ── Instantiate Controller ───────────────────────────────────────────────────
ctrl = Controller(mkt_B, mkt_C, demo_book, demo_mm, client_flow_fn)
print('Controller ready.')

Stock path: 24 steps, dt = 3600.0 s
Coefficient annualization : 77.76888838089432
Depth B: mean = 500,010 EUR, min = 497,047, max = 505,785
Depth C: mean = 200,006 EUR, min = 198,583, max = 202,816
Controller ready.


In [10]:
# ── Run the simulation ───────────────────────────────────────────────────────
import time
t0 = time.time()
ctrl.simulate()
elapsed = time.time() - t0

df_trades = ctrl.trade_history
df_log    = ctrl.step_log
mm_fills  = df_trades[~df_trades['is_hedge']]
hedges    = df_trades[df_trades['is_hedge']]

print(f'Simulation complete in {elapsed:.1f} s')
print(f'Steps logged        : {len(df_log)}')
print(f'Total trades        : {len(df_trades)}  '
      f'({len(mm_fills)} MM fills, {len(hedges)} hedge legs)')
print(f'Final inventory     : {ctrl.quoter.inventory:,.0f} EUR')
print(f'Quotes posted       : {ctrl._n_quotes_posted:,}')

KeyboardInterrupt: 

### Quick sanity checks — PnLTracker identity verification

Three mandatory identities from the spec:  
1. `realized + unrealized = total_mtm`  
2. `inception_spread + inventory_revaluation − fees = total_mtm`  
3. Sum of per-trade MtM columns = aggregate MtM at each row  

In [ ]:
current_mid = ctrl._current_fair_mid()
rep = PnLTracker.report(df_trades, current_mid)

# Identity 1: realized + unrealized == total_mtm
delta1 = abs(rep['realized_pnl'] + rep['unrealized_pnl'] - rep['total_mtm_pnl'])
assert delta1 < 1e-4, f'Identity 1 broken: delta = {delta1}'
print(f'[OK] Identity 1 — realized + unrealized == total_mtm  (delta = {delta1:.2e})')

# Identity 2: inception + revaluation - fees == total_mtm
delta2 = abs(rep['inception_spread_pnl'] + rep['inventory_revaluation_pnl']
             - rep['total_fees'] - rep['total_mtm_pnl'])
assert delta2 < 1e-4, f'Identity 2 broken: delta = {delta2}'
print(f'[OK] Identity 2 — inception + revaluation - fees == total_mtm  '
      f'(delta = {delta2:.2e})')

# Identity 3: per-trade MtM columns sum to aggregate MtM
if len(df_trades) > 0:
    aug = PnLTracker.augment(df_trades)
    ev  = PnLTracker.per_trade_mtm_evolution(df_trades)
    row_sums = ev.sum(axis=1).values
    max_delta3 = np.nanmax(np.abs(row_sums - aug['mtm_pnl'].values))
    assert max_delta3 < 1e-4, f'Identity 3 broken: max delta = {max_delta3}'
    print(f'[OK] Identity 3 — per-trade MtM sums == aggregate MtM  '
          f'(max delta = {max_delta3:.2e})')

print()
print('=== P&L Summary ===')
print(f'Total MtM P&L       : {rep["total_mtm_pnl"]:>+12.2f} USD')
print(f'  Realized          : {rep["realized_pnl"]:>+12.2f} USD')
print(f'  Unrealized        : {rep["unrealized_pnl"]:>+12.2f} USD')
print(f'  Inception spread  : {rep["inception_spread_pnl"]:>+12.2f} USD')
print(f'  Inv. revaluation  : {rep["inventory_revaluation_pnl"]:>+12.2f} USD')
print(f'Total fees          : {rep["total_fees"]:>12.2f} USD')
print(f'  Maker fees (A)    : {rep["mm_maker_fees"]:>12.2f} USD')
print(f'  Taker fees (B/C)  : {rep["hedge_taker_fees"]:>12.2f} USD')
print(f'Final inventory     : {rep["final_inventory_eur"]:>12.0f} EUR')
print(f'MM fills            : {rep["n_mm_fills"]:>12d}')
print(f'Hedge legs          : {rep["n_hedges"]:>12d}')

### Full backtesting report

`ctrl.report()` renders six panels:
1. Bid / Ask / Mid for markets A, B and C
2. Top 10 fills by size on market A
3. EUR/USD price vs inventory (± delta-limit lines)
4. 4-panel session P&L chart (PnLTracker)
5. Per-trade MtM percentiles (5th / median / 95th / mean)
6. Fill-rate analysis by ladder level

In [ ]:
ctrl.report()

### Supplementary — raw step-log snapshot

In [ ]:
display(ctrl.step_log.head(10))
print('...')
display(ctrl.step_log.tail(5))

In [ ]:
# ── MM fill detail (top 10 by size) ─────────────────────────────────────────
cols = ['step', 't', 'direction', 'level', 'price', 'size',
        'cash_flow', 'inventory_after', 'is_hedge', 'venue']
display(
    df_trades[~df_trades['is_hedge']]
    .nlargest(10, 'size')[cols]
    .reset_index(drop=True)
)